## GLP1 Data Cleaning for Analysis 
 Data downloaded from Prod_deid_clinical.ehr_dataset
- Medication -> glp1_orders.csv - all GLP1 prescriptions
- Flow_Sheet -> glp1_bmi.csv - BMI readings for all GLP1 patients
- Lab_result -> glp1_a1c.csv - HbA1C lab results for all GLP1 patients
- Patient -> glp1_patients.csv - patient demographics for all GLP1 patients
- Diagnosis -> glp1_t2d.csv- t2d diagnosis for all GLP1 patients 

Output Files: 
 - cohort_1 → cohort_5_final.csv - one row per patient, columns added at each step
 - clean_bmi.csv and clean_a1c.csv - all readings in the study window for analysis

Definitions: 
- baseline: most recent value in the 6 months before or on the start date 
- follow-up: most recent value between 6 and 12 months after the start date

In [1]:
# Import pandas needed for the script
import pandas as pd

In [2]:
# Read in the data files
glp1_t2d = pd.read_csv('/Users/jen/Desktop/HDM_26/GLP1 Final Project /glp1_t2d.csv')
glp1_a1c = pd.read_csv('/Users/jen/Desktop/HDM_26/GLP1 Final Project /glp1_a1c.csv')
glp1_bmi = pd.read_csv('/Users/jen/Desktop/HDM_26/GLP1 Final Project /glp1_bmi.csv')
glp1_orders = pd.read_csv('/Users/jen/Desktop/HDM_26/GLP1 Final Project /glp1_orders.csv', low_memory=False)
glp1_patients = pd.read_csv('/Users/jen/Desktop/HDM_26/GLP1 Final Project /glp1_patients.csv')

### Explore Data: 
### 1. GLP1 Orders Table

In [3]:
# Check columns in the medication orders file
glp1_orders.shape, glp1_orders.columns

((1581462, 20),
 Index(['pat_enc_csn_id', 'pat_id', 'ordering_time', 'order_start_time',
        'order_end_time', 'discontinue_time', 'sig', 'discrete_dose',
        'dose_unit', 'lastdose', 'frequency', 'route', 'medication_name',
        'generic_name', 'thera_class', 'pharm_class', 'pharm_subclass',
        'order_class', 'order_status', 'ordering_mode'],
       dtype='str'))

In [4]:
# Check values in 'thera_class' column to identify any unclassified/missing drug products
glp1_orders['thera_class'].value_counts(dropna=False)

thera_class
ANTIHYPERGLYCEMICS            917591
ANTI-OBESITY DRUGS            663430
UNCLASSIFIED DRUG PRODUCTS       440
NaN                                1
Name: count, dtype: int64

In [5]:
# Check which drug names are associated with each therapeutic class
pd.crosstab(glp1_orders['medication_name'].str.split().str[0].str.upper(),
            glp1_orders['thera_class'])


thera_class,ANTI-OBESITY DRUGS,ANTIHYPERGLYCEMICS,UNCLASSIFIED DRUG PRODUCTS
medication_name,,,
ADLYXIN,0,116,0
BYDUREON,0,5401,0
BYETTA,0,941,0
DULAGLUTIDE,0,24786,0
EXENATIDE,0,5064,0
INSULIN,0,667,0
IRB,0,0,440
LIRAGLUTIDE,22893,22327,0
LIXISENATIDE,0,151,0


In [6]:
# Check which are the "unclassified" drug names in the medication orders file
glp1_orders.loc[glp1_orders['thera_class'] == 'UNCLASSIFIED DRUG PRODUCTS', 'medication_name'].value_counts().head(10)

medication_name
IRB - SEMAGLUTIDE 0.25 MG OR PLACEBO/INVESTIGATIONAL INJECTION                          107
IRB - SEMAGLUTIDE 0.5 MG OR PLACEBO/INVESTIGATIONAL INJECTION                            87
IRB - SEMAGLUTIDE/PLACEBO/INVESTIGATIONAL 1.34 MG/ML INJECTION                           74
IRB - SEMAGLUTIDE 1.0 MG OR PLACEBO/INVESTIGATIONAL INJECTION                            70
IRB - SEMAGLUTIDE/PLACEBO/INVESTIGATIONAL INJ                                            41
IRB - SEMAGLUTIDE (B/D 0.25/0.5/1/1.7/2.4 MG) OR PLACEBO/INVESTIGATIONAL INJECTION       12
IRB - TIRZEPATIDE/PLACEBO/INVESTIGATIONAL 5 MG/0.5 ML INJECTION                           9
IRB - CAGRISEMA 1.0+0.5/CAGRILINTIDE 1.0/SEMAGLUTIDE 0.5/PLACEBO/INVESTIGATIONAL INJ      7
IRB - CAGRISEMA 4.0+2.0/CAGRILINTIDE 4.0/SEMAGLUTIDE 2.0/PLACEBO/INVESTIGATIONAL INJ      6
IRB - CAGRISEMA 9.6+4.8/CAGRILINTIDE 9.6/SEMAGLUTIDE 4.8/PLACEBO/INVESTIGATIONAL INJ      6
Name: count, dtype: int64

In [7]:
# Remove IRB and missing values from the medication orders file 
medication_orders_cleaned = glp1_orders[glp1_orders['thera_class'] != 'UNCLASSIFIED DRUG PRODUCTS']
medication_orders_cleaned = medication_orders_cleaned[medication_orders_cleaned['thera_class'].notna()]
medication_orders_cleaned['thera_class'].value_counts(dropna=False)

thera_class
ANTIHYPERGLYCEMICS    917591
ANTI-OBESITY DRUGS    663430
Name: count, dtype: int64

- We removed IRB orders because they are drugs given as part of a research study as noted in the output

In [8]:
# Remove insulin and GLP1 combination drugs 
medication_orders_cleaned = medication_orders_cleaned[~medication_orders_cleaned['medication_name'].str.upper().str.startswith(('INSULIN', 'SOLIQUA', 'XULTOPHY'))]
medication_orders_cleaned['thera_class'].value_counts()

thera_class
ANTIHYPERGLYCEMICS    914193
ANTI-OBESITY DRUGS    663430
Name: count, dtype: int64

- We removed insulin and GLP1 combination drugs because insulin is known to lower HbA1c and impacts weight. Including it in the study would make it hard to determine whether changes are due to a GLP1 or insulin

In [9]:
# Check order status 
medication_orders_cleaned['order_status'].value_counts(dropna=False)

order_status
Sent            1305462
NaN              256443
Suspend           13666
Discontinued       1539
Completed           510
Dispensed             3
Name: count, dtype: int64

- From the output there were no orders that were canceled and therefore none were removed. 

In [10]:
# Convert order_date to datetime format
medication_orders_cleaned['ordering_time'] = pd.to_datetime(medication_orders_cleaned['ordering_time'])

In [11]:
# Keep patients first GLP1 order only. This indicates the date of their treatement initiation.
medication_orders_cleaned = medication_orders_cleaned.sort_values('ordering_time', kind='stable')
glp1_first_orders = medication_orders_cleaned.drop_duplicates('pat_id', keep='first')
len(glp1_first_orders)

256154

- We are using each patients first GLP1 order as their treatment start date. The before/after measurements are anchored to this start date. We also kept one row per patient.

In [12]:
# Keep patients who started GLP-1 for weight loss only.
glp1_weight_loss_only = glp1_first_orders[glp1_first_orders['thera_class'] == 'ANTI-OBESITY DRUGS'].copy()
len(glp1_weight_loss_only)

95097

- We are using the thera_class = "ANTI-OBESITY DRUGS" to distinguish between GLP1 obesity vs. diabetes treatment. 

In [13]:
# Remove patients with no start date 
glp1_weight_loss_only = glp1_weight_loss_only.dropna(subset=['ordering_time'])
len(glp1_weight_loss_only)

94989

In [14]:
# Add start date for each patient for all of their orders
start_dates = glp1_weight_loss_only[['pat_id', 'ordering_time']].rename(columns={'ordering_time': 'start_date'})
orders = medication_orders_cleaned.merge(start_dates, on='pat_id')

In [15]:
# Keep orders from the first year after starting
orders['days_after_start'] = (orders['ordering_time'] - orders['start_date']).dt.days
orders_year1 = orders[(orders['days_after_start'] >= 0) & (orders['days_after_start'] <= 365)]

In [16]:
# Number of orders each patient had in the first year
glp1_weight_loss_only['n_orders'] = glp1_weight_loss_only['pat_id'].map(orders_year1.groupby('pat_id').size())

# Days from start to each patient's last order in the first year
glp1_weight_loss_only['days_to_last_order'] = glp1_weight_loss_only['pat_id'].map(orders_year1.groupby('pat_id')['days_after_start'].max())

In [17]:
# Group patients by how long they stayed on treatment
glp1_weight_loss_only['persistence'] = 'Single order'
glp1_weight_loss_only.loc[glp1_weight_loss_only['days_to_last_order'] >= 1, 'persistence'] = 'Less than 3 months'
glp1_weight_loss_only.loc[glp1_weight_loss_only['days_to_last_order'] >= 90, 'persistence'] = 'Between 3-6 months'
glp1_weight_loss_only.loc[glp1_weight_loss_only['days_to_last_order'] >= 180, 'persistence'] = 'More than 6 months'

glp1_weight_loss_only['persistence'].value_counts()

persistence
Single order          40789
More than 6 months    31885
Less than 3 months    12564
Between 3-6 months     9751
Name: count, dtype: int64

- We grouped patients by how long they kept getting prescriptions.
    - For each patient, we looked at all their GLP-1 orders in the first year after starting and found the date of their last order. 
- All patients started on a weight-loss GLP-1, but all of their later GLP-1 orders were counted, including diabetes-type ones to account for patients who started on a weight-loss GLP and later switched to a diabetes glp1.
- We used the date of the last order rather than the number of orders, because one order can include several refills. 

- Groups indicate that the "Last order was..."
    - Single order - On the start date (no later orders)
    - Less than 3 months - 1–89 days after starting
    - Between 3–6 months - 90–179 days after starting 
    - More than 6 months -  180–365 days after starting 

In [18]:
# Save the cleaned medication orders file to a new CSV file
glp1_weight_loss_only.to_csv('/Users/jen/Desktop/HDM_26/GLP1 Final Project /cohort_1_medication.csv', index=False)

### 2. Diabetes Table

We split patients into two groups 
- patients who already had type 2 diabeted when they started the GLP1 
- patients who did not have type 2 diabetes when they started a GLP1

In [19]:
# Look at the diabetes table
glp1_t2d.shape, glp1_t2d.columns

((5995180, 7),
 Index(['pat_enc_csn_id', 'pat_id', 'DX_TYPE', 'dx_name', 'icd10', 'entry_date',
        'status_comment'],
       dtype='str'))

In [20]:
# Show ICD10 codes for diabetes - filtered during SQL export to only include codes for type 2 diabetes
glp1_t2d['icd10'].value_counts()

icd10
E11.9       4329117
E11.65       612778
E11.22       129696
E11.69       125012
E11.8        104115
             ...   
E11.3539          5
E11.351           4
E11.349           4
E11.321           2
E11.331           1
Name: count, Length: 93, dtype: int64

In [21]:
# Convert Dates to datetime format
glp1_t2d['entry_date'] = pd.to_datetime(glp1_t2d['entry_date'])

In [22]:
# Find each patient's first type 2 diabetes diagnosis date
glp1_weight_loss_only['t2d_dx_date'] = glp1_weight_loss_only['pat_id'].map(glp1_t2d.groupby('pat_id')['entry_date'].min())

In [23]:
# Group patients- T2D counted if patient was diagnosed on or before the GLP-1 start date
glp1_weight_loss_only['group'] = 'Obesity only'
glp1_weight_loss_only.loc[glp1_weight_loss_only['t2d_dx_date'] <= glp1_weight_loss_only['ordering_time'], 'group'] = 'Obesity + T2D'

glp1_weight_loss_only['group'].value_counts()

group
Obesity only     90635
Obesity + T2D     4354
Name: count, dtype: int64

- Obesity only: patients with no diabetes diagnosis on/after starting a GLP1
- Obesity and T2D: patients with a diabetes diagnosis on/before starting GLP1

In [24]:
# Save the cleaned medication orders file with diabetes group to a new CSV file
glp1_weight_loss_only.to_csv('/Users/jen/Desktop/HDM_26/GLP1 Final Project /cohort_2_diabetes.csv', index=False)

### 3. BMI Table

We found each patient's BMI before starting the GLP-1, their BMI 6–12 months later, and the change between the two

In [25]:
# Look at the BMI table
glp1_bmi.shape, glp1_bmi.columns

((833522, 10),
 Index(['line', 'pat_enc_csn_id', 'pat_id', 'flo_meas_id', 'name',
        'display_name', 'observation_value', 'observation_comment', 'unit',
        'recorded_time'],
       dtype='str'))

In [26]:
# Confirm all rows are BMI values
glp1_bmi['name'].value_counts()

name
R BMI    833522
Name: count, dtype: int64

In [27]:
# Check what the values look like
glp1_bmi['observation_value'].head(20)

0     40.00
1     31.88
2     36.61
3     31.20
4      0.00
5     40.28
6     29.63
7     28.61
8     42.07
9     42.88
10    26.07
11    25.00
12    42.02
13    45.23
14     0.00
15     0.00
16    31.31
17     0.00
18    38.60
19    30.96
Name: observation_value, dtype: float64

In [28]:
# Check how many BMI values are 0 or unrealistic
(glp1_bmi['observation_value'] < 10).sum(), (glp1_bmi['observation_value'] > 100).sum()

(np.int64(97923), np.int64(134))

In [29]:
# Removes 0s and typos
bmi = glp1_bmi[(glp1_bmi['observation_value'] >= 10) & (glp1_bmi['observation_value'] <= 100)].copy()
len(glp1_bmi), len(bmi)

(833522, 733623)

- 97,923 readings were 0 meaning BMI was likely not recorded at that visit
- 134 readings were over 100, which is most likely a typo

In [30]:
# Convert recorded_time to datetime format
bmi['recorded_time'] = pd.to_datetime(bmi['recorded_time'])

In [31]:
# Keep cohort patients only and add their GLP-1 start date
bmi = bmi[bmi['pat_id'].isin(glp1_weight_loss_only['pat_id'])].copy()
bmi['start_date'] = bmi['pat_id'].map(glp1_weight_loss_only.set_index('pat_id')['ordering_time'])
bmi['days_since_start'] = (bmi['recorded_time'] - bmi['start_date']).dt.days

In [32]:
# Keep readings from 6 months before to 1 year after starting
bmi = bmi[(bmi['days_since_start'] >= -180) & (bmi['days_since_start'] <= 365)]
len(bmi)

42196

In [33]:
# Save all BMI readings 
bmi.to_csv('/Users/jen/Desktop/HDM_26/GLP1 Final Project /clean_bmi.csv', index=False)

In [34]:
# Baseline BMI - last reading on or before the start date
baseline_bmi = bmi[bmi['days_since_start'] <= 0].sort_values('recorded_time')
glp1_weight_loss_only['baseline_bmi'] = glp1_weight_loss_only['pat_id'].map(baseline_bmi.groupby('pat_id')['observation_value'].last())

In [35]:
# Follow-up BMI - latest reading between 6 months and 1 year after starting
post_bmi = bmi[(bmi['days_since_start'] >= 180) & (bmi['days_since_start'] <= 365)].sort_values('recorded_time')
glp1_weight_loss_only['bmi_followup'] = glp1_weight_loss_only['pat_id'].map(post_bmi.groupby('pat_id')['observation_value'].last())

In [36]:
# BMI change- follow-up minus baseline 
glp1_weight_loss_only['bmi_change'] = glp1_weight_loss_only['bmi_followup'] - glp1_weight_loss_only['baseline_bmi']
glp1_weight_loss_only[['baseline_bmi', 'bmi_followup', 'bmi_change']].describe()

,baseline_bmi,bmi_followup,bmi_change
count,10385.000000,6438.000000,1689.000000
mean,34.598187,33.650008,-1.902090
std,7.198602,7.427639,3.729626
min,13.090000,14.210000,-22.980000
25%,29.660000,28.332500,-3.970000
50%,33.480000,32.575000,-1.600000
75%,38.380000,37.700000,0.320000
max,82.990000,76.610000,15.170000


In [37]:
glp1_weight_loss_only.to_csv('/Users/jen/Desktop/HDM_26/GLP1 Final Project /cohort_3_bmi.csv', index=False)

### 4. HbA1c table

Same approach as BMI, using HbA1c (average blood sugar over the past ~3 months)


In [38]:
# Look at the HbA1c table
glp1_a1c.shape, glp1_a1c.columns

((1342992, 14),
 Index(['pat_enc_csn_id', 'pat_id', 'order_description', 'ordering_date',
        'result_time', 'component_name', 'component_base_name', 'numeric_value',
        'raw_value', 'reference_unit', 'result_flag', 'result_status',
        'lab_status', 'loinc_code'],
       dtype='str'))

In [39]:
# Confirm all rows are HbA1c tests
glp1_a1c['component_name'].value_counts()

component_name
HEMOGLOBIN A1C                      1113117
HEMOGLOBIN A1C EXT                   150836
HEMOGLOBIN A1C, POC                   78889
A1C                                      73
GLYCOSYLATED HEMOGLOBIN (HGBA1C)         48
A1C BLOOD GLUCOSE                        15
HB A1C DIABETIC ASSESMENT                12
HEMOGLOBIN A1C (ARUP)                     2
Name: count, dtype: int64

In [40]:
# Check what the values look like
glp1_a1c[['numeric_value', 'raw_value', 'reference_unit']].head(20)

,numeric_value,raw_value,reference_unit
0,6.9,6.9,%
1,5.2,5.2,%
2,6.5,6.5,%
3,5.4,5.4,%
4,6.8,6.8,%
5,5.7,5.7,%
6,5.6,5.6,%
7,5.5,5.5,%
8,6.6,6.6,% of total Hgb
9,6.0,6.0,%


In [41]:
# Check all units and how many values are missing
glp1_a1c['reference_unit'].value_counts(dropna=False), glp1_a1c['numeric_value'].isna().sum()

(reference_unit
 %                 1141927
 % of total Hgb     115918
 NaN                 73668
 Percent             11441
 %Hb                    35
 percent                 3
 Name: count, dtype: int64,
 np.int64(56))

In [42]:
# Keep realistic HbA1c values only (3-20%)
a1c = glp1_a1c[(glp1_a1c['numeric_value'] >= 3) & (glp1_a1c['numeric_value'] <= 20)].copy()
len(glp1_a1c), len(a1c)

(1342992, 1338580)

We kept values between 3% and 20%
-  Normal HbA1c is under 5.7%, diabetes is 6.5% and above, and real values almost never go below 4% or above 15%

In [43]:
# Convert result_time to datetime format
a1c['result_time'] = pd.to_datetime(a1c['result_time'])

In [44]:
# Keep cohort patients only and add their GLP-1 start date
a1c = a1c[a1c['pat_id'].isin(glp1_weight_loss_only['pat_id'])].copy()
a1c['start_date'] = a1c['pat_id'].map(glp1_weight_loss_only.set_index('pat_id')['ordering_time'])
a1c['days_since_start'] = (a1c['result_time'] - a1c['start_date']).dt.days

In [45]:
# Keep results from 6 months before to 1 year after starting
a1c = a1c[(a1c['days_since_start'] >= -180) & (a1c['days_since_start'] <= 365)]
len(a1c)

99655

In [46]:
# Save all HbA1c results
a1c.to_csv('/Users/jen/Desktop/HDM_26/GLP1 Final Project /clean_a1c.csv', index=False)

In [47]:
# Baseline HbA1c - last result on or before the start date
baseline_a1c = a1c[a1c['days_since_start'] <= 0].sort_values('result_time')
glp1_weight_loss_only['baseline_a1c'] = glp1_weight_loss_only['pat_id'].map(baseline_a1c.groupby('pat_id')['numeric_value'].last())

In [48]:
# Follow-up HbA1c - latest result between 6 months and 1 year after starting
post_a1c = a1c[(a1c['days_since_start'] >= 180) & (a1c['days_since_start'] <= 365)].sort_values('result_time')
glp1_weight_loss_only['a1c_followup'] = glp1_weight_loss_only['pat_id'].map(post_a1c.groupby('pat_id')['numeric_value'].last())

In [49]:
# HbA1c change 
glp1_weight_loss_only['a1c_change'] = glp1_weight_loss_only['a1c_followup'] - glp1_weight_loss_only['baseline_a1c']
glp1_weight_loss_only[['baseline_a1c', 'a1c_followup', 'a1c_change']].describe()

,baseline_a1c,a1c_followup,a1c_change
count,38315.000000,19311.000000,11951.000000
mean,5.542490,5.412449,-0.200803
std,0.572018,0.552058,0.459752
min,4.000000,3.900000,-8.000000
25%,5.200000,5.100000,-0.400000
50%,5.500000,5.300000,-0.200000
75%,5.800000,5.600000,0.000000
max,14.200000,14.400000,6.600000


In [50]:
# HbA1c change by group
glp1_weight_loss_only.groupby('group')['a1c_change'].describe()

,count,mean,std,min,25%,50%,75%,max
group,,,,,,,,
Obesity + T2D,1172.0,-0.371758,0.948519,-8.0,-0.6,-0.2,0.1,4.4
Obesity only,10779.0,-0.182215,0.364807,-6.4,-0.4,-0.2,0.0,6.6


In [51]:
glp1_weight_loss_only.to_csv('/Users/jen/Desktop/HDM_26/GLP1 Final Project /cohort_4_a1c.csv', index=False)

### 5. Patient Table

We added age, sex and race to each patient so results can be compared across groups, and we kept adults only

In [52]:
# Look at the patient table
glp1_patients.shape, glp1_patients.columns

((256894, 7),
 Index(['pat_id', 'age', 'sex', 'ethnicity', 'race', 'living_status',
        'year_of_death'],
       dtype='str'))

In [53]:
# Check the values in each demographic column
glp1_patients['age'].value_counts(dropna=False)

age
63.0    6867
62.0    6822
65.0    6731
64.0    6641
61.0    6621
        ... 
13.0      47
12.0      11
11.0       3
10.0       2
9.0        1
Name: count, Length: 81, dtype: int64

In [54]:
glp1_patients['sex'].value_counts(dropna=False)

sex
Female     159650
Male        97143
Unknown       100
NaN             1
Name: count, dtype: int64

In [55]:
glp1_patients['race'].value_counts(dropna=False)

race
White                                                                                                                                                              144808
Black or African American                                                                                                                                           32984
Not listed                                                                                                                                                          16912
Hispanic or Latino or Spanish                                                                                                                                       15022
Don't know                                                                                                                                                          11616
                                                                                                                                                 

In [56]:
glp1_patients['ethnicity'].value_counts(dropna=False)

ethnicity
Not of Spanish/Hispanic Origin         123986
Don't know                              25168
NaN                                     22133
Not listed                              11947
Puerto Rican                             8817
                                        ...  
Jamaican, Cuban, African American           1
Haitian, Black, African American            1
Italian, Irish, German, Japanese            1
Haitian, Jamaican, African American         1
Romanian, Polish, Turkish, Jewish           1
Name: count, Length: 1438, dtype: int64

In [57]:
glp1_patients['living_status'].value_counts(dropna=False)

living_status
Alive            253987
Deceased           2904
Not A Patient         3
Name: count, dtype: int64

In [58]:
# Replace missing sex values with 'Unknown'
glp1_patients['sex'] = glp1_patients['sex'].fillna('Unknown')

In [59]:
# Group race into main categories
race_groups = {
    'White': 'White',
    'Black or African American': 'Black',
    'Hispanic or Latino or Spanish': 'Hispanic/Latino',
    'Asian': 'Asian',
    'Asian Indian': 'Asian',
    'Asian Not listed': 'Asian',
    'Not listed Asian': 'Asian',
    'Asian Indian Not listed': 'Asian',
    'Not listed Asian Indian': 'Asian',
    'Asian Indian Don\'t know': 'Asian',
    "Don't know": 'Unknown',
    'Prefer not to answer': 'Unknown'
}

# Combinations of races (e.g. "White Asian") are grouped as "Other/Multiracial"
glp1_patients['race_group'] = glp1_patients['race'].map(race_groups).fillna('Other/Multiracial')
glp1_patients.loc[glp1_patients['race'].isna(), 'race_group'] = 'Unknown'
glp1_patients['race_group'].value_counts()

race_group
White                144808
Black                 32984
Other/Multiracial     30288
Unknown               25278
Hispanic/Latino       15022
Asian                  8514
Name: count, dtype: int64

In [60]:
# Add demographics to each cohort patient
patients = glp1_patients.set_index('pat_id')
glp1_weight_loss_only['age'] = glp1_weight_loss_only['pat_id'].map(patients['age'])
glp1_weight_loss_only['sex'] = glp1_weight_loss_only['pat_id'].map(patients['sex'])
glp1_weight_loss_only['race_group'] = glp1_weight_loss_only['pat_id'].map(patients['race_group'])
glp1_weight_loss_only['living_status'] = glp1_weight_loss_only['pat_id'].map(patients['living_status'])

In [61]:
# Convert age to numeric values 
glp1_weight_loss_only['age'] = pd.to_numeric(glp1_weight_loss_only['age'], errors='coerce')

In [62]:
# Create age groups for analysis
glp1_weight_loss_only['age_group'] = 'Unknown'
glp1_weight_loss_only.loc[glp1_weight_loss_only['age'] >= 18, 'age_group'] = '18-39'
glp1_weight_loss_only.loc[glp1_weight_loss_only['age'] >= 40, 'age_group'] = '40-64'
glp1_weight_loss_only.loc[glp1_weight_loss_only['age'] >= 65, 'age_group'] = '65+'

In [63]:
# Filter out patients under 18 years old 
glp1_weight_loss_only = glp1_weight_loss_only[glp1_weight_loss_only['age'] >= 18]
len(glp1_weight_loss_only)

94477

In [64]:
glp1_weight_loss_only.to_csv('/Users/jen/Desktop/HDM_26/GLP1 Final Project /cohort_5_final.csv', index=False)

In [65]:
# Final counts for the cohort summary
print('Final cohort:', len(glp1_weight_loss_only))
print(glp1_weight_loss_only['group'].value_counts(), '\n')
print('With BMI change:', glp1_weight_loss_only['bmi_change'].notna().sum())
print('With HbA1c change:', glp1_weight_loss_only['a1c_change'].notna().sum())

Final cohort: 94477
group
Obesity only     90146
Obesity + T2D     4331
Name: count, dtype: int64 

With BMI change: 1682
With HbA1c change: 11887
